In [2]:
pip install statsmodels

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 9.8 MB/s  0:00:016m0:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [statsmodels] [statsmodels]
Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.metrics import confusion_matrix, classification_report

from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from sklearn.tree import (DecisionTreeClassifier as DTC,
                          DecisionTreeRegressor as DTR,
                          plot_tree,
                          export_text)
from sklearn.metrics import (accuracy_score,
                             log_loss)
from sklearn.ensemble import \
     (RandomForestRegressor as RF,
      GradientBoostingRegressor as GBR)


In [4]:
df = pd.read_csv('../../Data/Processed/Final_Result_All_Signals_Combined.csv', index_col=0, parse_dates=True)
df.head()
# Replace 'malignant' with 1 and 'benign' with 0
df = df.query("condition != 'black'")
# Multiple categories become 1
df['condition'] = df['condition'].replace({
    'neutral': 0,
    'positive': 1,
    'negative': 1
})
print(df['condition'].value_counts(dropna=False))

/var/folders/zc/kzlmzxqx1bx13blggjybvfx1xkbyfl/T/ipykernel_34270/3288157032.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df = pd.read_csv('../../Data/Processed/Final_Result_All_Signals_Combined.csv', index_col=0, parse_dates=True)


condition
1    276
0    138
Name: count, dtype: int64


/var/folders/zc/kzlmzxqx1bx13blggjybvfx1xkbyfl/T/ipykernel_34270/3288157032.py:6: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['condition'] = df['condition'].replace({


In [ ]:
df.head()

,X,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,skewness_raw,mean_raw,Mean_Tonic,Max_Tonic,Min_Tonic,SD_Tonic,Mean_Phasic,Max_Phasic,Min_Phasic,SD_Phasic
Unnamed: 0,,,,,,,,,,,,,,,,,,,,,
1,SUD_003_01_EDA_10s,0.356146,0.361270,0.354865,0.354865,0.353584,0.353584,0.352302,0.352302,0.351021,...,-3.042697,0.318193,-0.024601,0.358658,-0.362070,0.237248,0.343182,0.695387,0.000000,0.228815
2,SUD_003_01_EDA_20s,0.333086,0.330524,0.331805,0.329243,0.330524,0.329243,0.329243,0.329243,0.327962,...,-0.078770,0.321844,-0.161865,0.155737,-0.370690,0.176927,0.483110,0.701306,0.154895,0.183622
3,SUD_003_01_EDA_30s,0.307464,0.310026,0.308745,0.308745,0.308745,0.307464,0.307464,0.307464,0.307464,...,-0.193585,0.302211,0.268743,0.295805,0.169441,0.033962,0.033986,0.139749,0.006041,0.035939
4,SUD_003_01_EDA_40s,0.294653,0.294653,0.293372,0.293372,0.295934,0.292091,0.292091,0.292091,0.292091,...,-0.095485,0.287799,0.266628,0.272488,0.262885,0.003119,0.020351,0.027083,0.006031,0.006480
5,SUD_003_01_EDA_50s,0.279280,0.277999,0.276718,0.275436,0.276718,0.275436,0.275436,0.275436,0.272874,...,0.792911,0.271977,0.270931,0.273679,0.265406,0.002695,0.001705,0.005345,0.000305,0.001209


In [ ]:
#Do not do log, do z score for all variables
from sklearn.preprocessing import StandardScaler

# 1. Get a list of all columns EXCEPT 'condition'
cols_to_scale = df.columns.difference(['condition', 'source_rows', 'X'])

# 2. Initialize and apply the scaler only to those columns
scaler = StandardScaler()
df[cols_to_scale] = scaler.fit_transform(df[cols_to_scale])

# Now df has standardized features, but 'condition' remains 0/1 (or its original state)

In [ ]:
import pandas as pd

# 1. Define which columns are features (X)
# This excludes 'condition', 'source_rows', and the 'X' identifier column so the model can't "cheat"
cols_to_scale = df.columns.difference(['condition', 'source_rows', 'X'])
X = df[cols_to_scale]

# 2. Define your target (y)
# Fixed the typo from 'condtion' to 'condition'
y = df['condition']

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.metrics import confusion_matrix, classification_report
import timeit
start_time = timeit.default_timer()

condition_col = 'condition'  # rename the variable to avoid collision
colnames = df.drop(columns=[condition_col, 'source_rows']).columns.tolist()

formula = f'{condition_col} ~ {" + ".join(colnames)}'
model = smf.glm(formula=formula, data=df, family=sm.families.Binomial())

result = model.fit()
print(result.summary())

# Print computational time and accuracy
elapsed = timeit.default_timer() - start_time
print("--- Run time is %s seconds ---" % elapsed)

/usr/local/lib/python3.12/dist-packages/statsmodels/genmod/generalized_linear_model.py:1342: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  warnings.warn(msg, category=PerfectSeparationWarning)
/usr/local/lib/python3.12/dist-packages/statsmodels/genmod/generalized_linear_model.py:1342: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  warnings.warn(msg, category=PerfectSeparationWarning)
/usr/local/lib/python3.12/dist-packages/statsmodels/genmod/generalized_linear_model.py:1342: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  warnings.warn(msg, category=PerfectSeparationWarning)
/usr/local/lib/python3.12/dist-packages/statsmodels/genmod/generalized_linear_model.py:1342: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  warnings.warn(msg, category=PerfectSeparationWarning)


                 Generalized Linear Model Regression Results                  
Dep. Variable:              condition   No. Observations:                  368
Model:                            GLM   Df Residuals:                        0
Model Family:                Binomial   Df Model:                          367
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:            -2.9013e-09
Date:                Tue, 24 Mar 2026   Deviance:                   5.8034e-09
Time:                        12:35:06   Pearson chi2:                 2.90e-09
No. Iterations:                    24   Pseudo R-squ. (CS):             0.7112
Covariance Type:            nonrobust                                         
                               coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------
Intercept               

In [ ]:
import lightgbm as lgb
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
import timeit

# Define start time to calculate computation time
start_time = timeit.default_timer()

y_train = df['condition']
Y = df['condition']

# Initialize the LightGBM model
model = lgb.LGBMClassifier(learning_rate=0.09, max_depth=-5, random_state=42)

# Train the model
model.fit(X.values, y_train.values, eval_set=[(X.values, Y.values)], eval_metric='logloss')

# Predict on the test set
y_pred = model.predict(X.values)

# Calculate accuracy
accuracy = accuracy_score(Y, y_pred)


# Print computational time and accuracy
elapsed = timeit.default_timer() - start_time
print("--- Run time is %s seconds ---" % elapsed)
print(f"Accuracy on the test set: {accuracy:.2f}")

print(classification_report(Y, y_pred, digits=3))


[LightGBM] [Info] Number of positive: 276, number of negative: 138
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000525 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 6082
[LightGBM] [Info] Number of data points in the train set: 414, number of used features: 54
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.666667 -> initscore=0.693147
[LightGBM] [Info] Start training from score 0.693147
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best 

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [ ]:
from sklearn.model_selection import train_test_split
import timeit
import lightgbm as lgb
from sklearn.metrics import accuracy_score, classification_report

# Define start time to calculate computation time
start_time = timeit.default_timer()

# Perform train-test split
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42, stratify=Y)

# Initialize the LightGBM model
# Keeping the same parameters as before for consistency
model_split = lgb.LGBMClassifier(learning_rate=0.09, max_depth=-5, random_state=42)

# Train the model on the training set
model_split.fit(X_train, y_train, eval_set=[(X_test, y_test)], eval_metric='logloss')

# Predict on the test set
y_pred_split = model_split.predict(X_test)

# Calculate accuracy on the test set
accuracy_split = accuracy_score(y_test, y_pred_split)

# Print computational time and accuracy
elapsed_split = timeit.default_timer() - start_time
print("--- Run time for train-test split model is %s seconds ---" % elapsed_split)
print(f"Accuracy on the test set (with split): {accuracy_split:.2f}")

# Print classification report for the test set
print(classification_report(y_test, y_pred_split, digits=3))


[LightGBM] [Info] Number of positive: 221, number of negative: 110
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000361 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5003
[LightGBM] [Info] Number of data points in the train set: 331, number of used features: 54
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.667674 -> initscore=0.697682
[LightGBM] [Info] Start training from score 0.697682
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best 

In [ ]:
import pandas as pd

feat_imp = pd.DataFrame({
    'feature': X_train.columns,
    'importance': model_split.feature_importances_
}).sort_values('importance', ascending=False)

print(feat_imp)

         feature  importance
5     Min_Phasic         138
44           V40          96
52  skewness_raw          87
9       SD_Tonic          86
50  kurtosis_raw          69
53       std_raw          57
6      Min_Tonic          50
3    Mean_Phasic          36
1      Max_Tonic          33
7        Min_raw          32
4     Mean_Tonic          29
14           V13          26
30           V28          24
40           V37          23
41           V38          22
2        Max_raw          21
36           V33          20
26           V24          19
42           V39          19
47            V7          18
28           V26          18
35           V32          18
39           V36          17
34           V31          17
0     Max_Phasic          15
8      SD_Phasic          15
13           V12          14
37           V34          14
31           V29          13
38           V35          13
12           V11          13
10            V1          12
27           V25          11
18           V